In [7]:
#A final check on whether the dataset has any duplicate rows

import unicodedata
import pandas as pd
from google.colab import files

COLUMN_NAME = None
SHEET_NAME = 0
NORMALIZE = True

OUTPUT_FILE = "Duplicate_Report.xlsx"


# Invisible characters that often differ between visually identical texts
ZERO_WIDTH = dict.fromkeys(map(ord, "\u200b\u200c\u200d\u2060\ufeff"), None)


def normalize(text):
    """Make two visually identical Bangla comments compare as equal."""
    if pd.isna(text):
        return None
    t = str(text)
    if NORMALIZE:
        t = unicodedata.normalize("NFC", t).translate(ZERO_WIDTH).lower()

        t = "".join(
            " " if unicodedata.category(ch)[0] in ("P", "S") else ch for ch in t
        )
        t = " ".join(t.split())
    else:
        t = t.strip()
    return t or None

**Uploading the Excel file**

In [8]:
print("Please upload the Excel file: ")
uploaded = files.upload()
input_path = next(iter(uploaded))

Please upload the Excel file: 


Saving Final Dataset for Duplication Checking.xlsx to Final Dataset for Duplication Checking.xlsx


**Reading the Excel file**

In [9]:
df = pd.read_excel(input_path, sheet_name=SHEET_NAME, dtype=str)

col = COLUMN_NAME if COLUMN_NAME else df.columns[0]
if col not in df.columns:
    raise ValueError(f"Column '{col}' not found. Available columns: {list(df.columns)}")
print(f"Using column: '{col}'  |  Total rows: {len(df)}")

Using column: 'Cleaned Text'  |  Total rows: 3747


**Finding duplicate rows {Dataframe index+2 since row 1 is the header}**

In [10]:
df["_key"] = df[col].apply(normalize)
df["Excel_Row"] = df.index + 2
is_dup = df["_key"].notna() & df["_key"].duplicated(keep=False)
dup = df[is_dup].copy()

**Building report tables**

In [11]:
summary_rows, detail_rows = [], []
group_id = 0

for key, g in dup.groupby("_key", sort=False):
    group_id += 1
    rows = g["Excel_Row"].tolist()
    rows_str = ", ".join(map(str, rows))
    summary_rows.append({
        "Group ID": group_id,
        "Comment": g[col].iloc[0],
        "Times Repeated": len(rows),
        "Row Numbers": rows_str,
    })
    for r, original in zip(rows, g[col]):
        detail_rows.append({
            "Group ID": group_id,
            "Row Number": r,
            "Comment": original,
            "Other Rows With Same Comment": ", ".join(str(x) for x in rows if x != r),
        })

summary = pd.DataFrame(summary_rows, columns=["Group ID", "Comment", "Times Repeated", "Row Numbers"])
detail = pd.DataFrame(detail_rows, columns=["Group ID", "Row Number", "Comment", "Other Rows With Same Comment"])

# Original data, untouched, with flag columns added at the end
annotated = df.drop(columns=["_key", "Excel_Row"]).copy()
group_lookup = {r["Row Number"]: r["Group ID"] for r in detail_rows}
annotated["Is Duplicate"] = [("Yes" if (i + 2) in group_lookup else "No") for i in df.index]
annotated["Duplicate Group ID"] = [group_lookup.get(i + 2, "") for i in df.index]

**Saving the Output File**

In [12]:
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="Summary", index=False)
    detail.to_excel(writer, sheet_name="Duplicate Rows", index=False)
    annotated.to_excel(writer, sheet_name="All Data (flagged)", index=False)
    for ws in writer.book.worksheets:
        for column_cells in ws.columns:
            letter = column_cells[0].column_letter
            ws.column_dimensions[letter].width = min(
                60, max(12, max(len(str(c.value or "")) for c in column_cells[:200]) + 2)
            )

print(f"\nDuplicate groups: {len(summary)}")
print(f"Rows involved in duplication: {len(detail)}")
print(f"Report saved as: {OUTPUT_FILE}")
files.download(OUTPUT_FILE)


Duplicate groups: 12
Rows involved in duplication: 25
Report saved as: Duplicate_Report.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>